# Tiny LLM — DPO (Direct Preference Optimization) Walkthrough

**Prerequisites:** An SFT checkpoint (e.g. `checkpoints/sft/best.pt`). DPO is phase 2 — it consumes an SFT-tuned model.

```
Pretrained checkpoint → SFT → DPO
                              ^^^^ this notebook
```

## 0. Setup

In [ ]:
import os, pathlib, sys

_cwd = pathlib.Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').exists() and (p / 'src').exists()),
    _cwd,
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print(f'Working directory: {REPO_ROOT}')

SFT_CHECKPOINT = 'checkpoints/sft/best.pt'
DPO_CONFIG     = 'configs/dpo.toml'
DEVICE         = 'auto'
TRAINING_MODE  = 'smoke'  # 'smoke' | 'quick' | 'full'

### Verify SFT checkpoint

In [ ]:
ckpt_path = pathlib.Path(SFT_CHECKPOINT)
if not ckpt_path.exists():
    sft_dir = pathlib.Path('checkpoints/sft')
    if sft_dir.exists():
        print(f'Available: {[p.name for p in sorted(sft_dir.glob("*.pt"))]}')
    raise FileNotFoundError(f'{SFT_CHECKPOINT} not found — run SFT first')

import torch
meta = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
mc = meta.get('model_config')
print(f'Checkpoint: {ckpt_path}  ({ckpt_path.stat().st_size / 1e6:.1f} MB)')
print(f'chat_template: {mc.get("chat_template", "not set") if mc else "no model_config"}')
del meta

## 1. Download & Tokenize DPO Dataset

Default: **argilla/distilabel-intel-orca-dpo-pairs** (prompt + chosen + rejected triples).

In [ ]:
from tiny_llm.posttraining.dpo_dataset import prepare_dpo_dataset
from tiny_llm.utils.config import load_config, merge_cli_overrides

overrides = []
if TRAINING_MODE == 'smoke':
    overrides = ['posttraining.max_samples=200', 'posttraining.epochs=1']
elif TRAINING_MODE == 'quick':
    overrides = ['posttraining.epochs=1']

cfg = merge_cli_overrides(load_config(DPO_CONFIG), overrides)
post_cfg = cfg.posttraining.model_dump()

train_examples, val_examples = prepare_dpo_dataset(
    datasets=post_cfg['dpo_datasets'],
    cache_dir=post_cfg.get('dpo_dataset_dir', 'data/dpo'),
    max_seq_len=int(post_cfg['max_seq_len']),
    val_fraction=float(post_cfg['val_fraction']),
    max_samples=post_cfg.get('max_samples'),
    seed=42,
)
print(f'Train: {len(train_examples):,}  Val: {len(val_examples):,}')

## 2. Inspect Examples

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')

for i, ex in enumerate(train_examples[:3]):
    print(f'=== Example {i} ===')
    print(f'Prompt ({len(ex.prompt_ids)} tok): {enc.decode(ex.prompt_ids)[:200]}...')
    print(f'Chosen ({len(ex.chosen_ids)} tok): {enc.decode(ex.chosen_ids)[:150]}...')
    print(f'Rejected ({len(ex.rejected_ids)} tok): {enc.decode(ex.rejected_ids)[:150]}...')
    print()

## 3. Run DPO Training

Key hyperparams: `beta=0.1`, `lr=5e-6`, `epochs=1`.

In [ ]:
import subprocess, sys

cmd = [
    'uv', 'run', 'python', 'main.py',
    '--stage', 'posttraining',
    '--config', DPO_CONFIG,
    '--checkpoint', SFT_CHECKPOINT,
    '--device', DEVICE,
]
if overrides:
    cmd += ['--overrides'] + overrides
if not pathlib.Path('data/tokenized/val.bin').exists():
    cmd += ['posttraining.pretrain_val_path=']

print(' '.join(cmd))
proc = subprocess.Popen(
    cmd, cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
if proc.wait() != 0:
    raise RuntimeError(f'DPO exited with code {proc.returncode}')
print('Done!')

## 4. Training Plots

In [ ]:
from IPython.display import Image, display

for name in ['training_quicklook.png', 'training_detailed.png']:
    p = pathlib.Path(f'checkpoints/dpo/plots/{name}')
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

## 5. Metrics

In [ ]:
import pandas as pd

mp = pathlib.Path('checkpoints/dpo/metrics.csv')
if mp.exists():
    df = pd.read_csv(mp)
    cols = [c for c in ['step', 'train/loss', 'train/accuracy',
                        'train/implicit_reward_margin', 'val/loss', 'val/accuracy']
            if c in df.columns]
    display(df[cols].dropna(subset=['train/loss']).tail(10))

## 6. Inference with DPO Checkpoint

In [ ]:
import glob

best_pts = sorted(glob.glob('checkpoints/dpo/best_*.pt'))
step_pts = sorted(glob.glob('checkpoints/dpo/step_*.pt'))
dpo_ckpt = best_pts[-1] if best_pts else (step_pts[-1] if step_pts else None)

if dpo_ckpt:
    for p in ['Explain neural networks simply.', 'Write a haiku about gradient descent.']:
        print(f'\n--- {p} ---')
        r = subprocess.run(
            ['uv', 'run', 'python', 'main.py', '--stage', 'inference',
             '--checkpoint', dpo_ckpt, '--chat-template', 'alpaca',
             '--prompt', p, '--max-tokens', '120', '--temperature', '0.7',
             '--device', DEVICE],
            cwd=str(REPO_ROOT), capture_output=True, text=True)
        print(r.stdout.strip())
else:
    print('No DPO checkpoint found.')

## 7. Compare SFT vs DPO

In [ ]:
def gen(ckpt, prompt):
    r = subprocess.run(
        ['uv', 'run', 'python', 'main.py', '--stage', 'inference',
         '--checkpoint', ckpt, '--chat-template', 'alpaca',
         '--prompt', prompt, '--max-tokens', '100', '--temperature', '0.7',
         '--device', DEVICE],
        cwd=str(REPO_ROOT), capture_output=True, text=True)
    return r.stdout.strip()

if dpo_ckpt and pathlib.Path(SFT_CHECKPOINT).exists():
    for p in ['Explain gradient descent.', 'What is the capital of France?']:
        print(f'\nPrompt: {p}')
        print(f'SFT:  {gen(SFT_CHECKPOINT, p)}')
        print(f'DPO:  {gen(dpo_ckpt, p)}')

---
## CLI Quick Reference
```bash
# Smoke test
uv run python main.py --stage posttraining --config configs/dpo.toml \
    --checkpoint checkpoints/sft/best.pt \
    --overrides posttraining.max_samples=200 posttraining.epochs=1

# Full run
uv run python main.py --stage posttraining --config configs/dpo.toml \
    --checkpoint checkpoints/sft/best.pt --device cuda
```